# Clasificación binaria de reingreso hospitalario a 30 días

**Micro-proyecto · Desarrollo de Soluciones · MAIA, Universidad de los Andes — Entrega 2**

Leonardo Almanza Sánchez

## Introducción

### Descripción

Este cuaderno documenta el desarrollo y la evaluación del modelo de clasificación que
responde la pregunta de negocio del proyecto:

> **¿Qué pacientes diabéticos van a reingresar al hospital dentro de los 30 días siguientes al alta?**

La respuesta alimenta una decisión concreta que se toma el día del alta: **a qué pacientes se
les programa control de seguimiento**, cuando la capacidad de hacerlo es limitada.

El modelo entrega dos salidas por paciente:

| Salida | Uso |
|---|---|
| `predict(X)` → 0 o 1 | la clasificación: reingresa o no |
| `predict_proba(X)` → probabilidad | dato de referencia para que el tablero filtre el listado |

#### Arquitectura del clasificador

1. Entrenar un clasificador binario sobre datos clínicos fuertemente desbalanceados (11,4 % de positivos).
2. Contrastar seis técnicas de balanceo de clases aplicadas **sin producir fuga de datos**.
3. Elegir la métrica de evaluación a partir del costo real de cada tipo de error.
4. Interpretar la matriz de confusión en términos operativos, no estadísticos.

### Teoría

#### Por qué la exactitud no sirve

De los 99 343 egresos del conjunto de trabajo, solo el 11,4 % termina en reingreso temprano.
Un clasificador que responda siempre «no reingresa» acierta el **88,6 %** de las veces sin
identificar a un solo paciente en riesgo. La exactitud premia ese comportamiento.

#### Los dos errores no cuestan lo mismo

| Error | Qué pasa en la práctica | Costo operativo |
|---|---|---|
| **Falso positivo** — se dice «va a reingresar» y no lo hace | Se le asigna seguimiento a alguien que no lo necesitaba tanto | Se gasta un cupo de la capacidad limitada. El costo es de oportunidad y el daño es relativamente bajo. |
| **Falso negativo** — se dice «no va a reingresar» y sí lo hace | El paciente no recibe seguimiento y después reingresa | Se pierde la oportunidad de intervenir: cama ocupada, costos hospitalarios altos, posible empeoramiento del paciente y más carga al sistema. |

El falso negativo es el error caro. Por eso la métrica principal de este trabajo es **F2**, que
pesa el doble la sensibilidad sobre la precisión:

$$F_\beta = (1+\beta^2)\cdot\frac{\text{precisión}\cdot\text{sensibilidad}}{\beta^2\cdot\text{precisión}+\text{sensibilidad}}, \qquad \beta=2$$

Es el mismo criterio que adoptan Nunes et al. (2025) al trabajar con datos clínicos desbalanceados.

#### El riesgo de fuga al balancear

Las técnicas de remuestreo generan o descartan observaciones. Si se aplican **antes** de separar
entrenamiento y prueba, ejemplos sintéticos derivados de pacientes de prueba terminan en el
conjunto de entrenamiento, y el desempeño medido deja de ser real. Es la explicación más probable
de los F1 cercanos a 1,00 que aparecen en parte de la literatura sobre este tipo de problemas.


#### Inicialización

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.features import construccion as cons, esquema as esq
from src.models import particion as part, metricas as met, entrenamiento as ent

pd.set_option("display.width", 160)
sns.set_theme(style="white")

SUPERFICIE, TINTA, SERIE, ALERTA = "#fcfcfb", "#0b0b0b", "#2a78d6", "#c8532b"
plt.rcParams.update({"figure.facecolor": SUPERFICIE, "axes.facecolor": SUPERFICIE,
                     "font.size": 9, "axes.titlesize": 10})

# Servidor de MLflow del proyecto. Cambiar por la IP vigente de la EC2:
# la instancia usa IP elastica y cambia al reiniciarse.
URI_MLFLOW = "http://3.215.79.195:8050"

print("criterio de umbral :", esq.CRITERIO_UMBRAL)
print("semilla            :", esq.SEMILLA)

#### Cargar los datos

La lectura desactiva la interpretación automática de nulos de pandas. El archivo marca el dato
ausente con `?`, y las columnas `A1Cresult` y `max_glu_serum` traen el texto literal `None`, que
**no** significa dato faltante sino que el examen no se ordenó — información clínica que el EDA
de la Entrega 1 identificó y que no debe destruirse (EDA 1.2).

In [ ]:
crudos = cons.cargar()
fallas = cons.verificar_carga(crudos)
print("comprobaciones fallidas:", fallas or "ninguna")
print(f"registros crudos: {len(crudos):,}  columnas: {crudos.shape[1]}")

Se aplican las exclusiones definidas en el EDA 5.2 y 5.3: los egresos por fallecimiento, que no
admiten reingreso, y los egresos a hospicio, donde programar un control no es la intervención
pertinente.

In [ ]:
trabajo = cons.conjunto_de_trabajo(crudos)

print(f"conjunto de trabajo : {len(trabajo):,} egresos")
print(f"pacientes distintos : {trabajo['patient_nbr'].nunique():,}")
print(f"tasa de reingreso   : {trabajo[esq.OBJETIVO].mean():.4f}")

#### Visualización de datos

In [ ]:
fig, ejes = plt.subplots(1, 2, figsize=(9.6, 3.2))

conteo = trabajo[esq.OBJETIVO].value_counts().sort_index()
ejes[0].bar(["No reingresa", "Reingresa < 30 d"], conteo.values,
            color=["#a8c8ec", SERIE], width=0.6)
for i, v in enumerate(conteo.values):
    ejes[0].text(i, v + 900, f"{v:,}".replace(",", " ") + f"\n{v/len(trabajo):.1%}",
                 ha="center", fontsize=8.5)
ejes[0].set_ylim(0, conteo.max() * 1.22)
ejes[0].set_yticks([])
ejes[0].set_title("Desbalance de la variable objetivo", loc="left")

tramos = pd.cut(trabajo["number_inpatient"], [-1, 0, 1, 2, 4, 10**6],
                labels=["0", "1", "2", "3-4", "5 o mas"])
tasa = trabajo.groupby(tramos, observed=True)[esq.OBJETIVO].mean() * 100
ejes[1].bar(range(len(tasa)), tasa.values, color=SERIE, width=0.6)
for i, v in enumerate(tasa.values):
    ejes[1].text(i, v + 0.7, f"{v:.1f} %", ha="center", fontsize=8.5)
ejes[1].axhline(trabajo[esq.OBJETIVO].mean() * 100, color="#9a9992",
                linestyle=(0, (4, 3)), linewidth=1)
ejes[1].set_xticks(range(len(tasa)), tasa.index)
ejes[1].set_xlabel("Hospitalizaciones en el año previo")
ejes[1].set_title("La señal más fuerte del conjunto (EDA 7.1)", loc="left")

for ax in ejes:
    ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
plt.show()

La clase positiva es minoritaria en una proporción de casi ocho a uno. El panel derecho muestra
por qué el problema es aprendible pese a ello: el historial de hospitalizaciones del año previo
separa la tasa de reingreso desde 8,6 % hasta 37,1 %, un gradiente monótono de más de cuatro veces.

#### División de los conjuntos de entrenamiento

La partición **no** es aleatoria por fila. El conjunto tiene 16 341 pacientes con más de un
encuentro, y el 46 % de las filas les pertenece. Una partición aleatoria dejaría encuentros del
mismo paciente a ambos lados y comprometería el 41,5 % de las filas de evaluación (EDA 8.1).

La razón es circular: si un paciente reingresa antes de 30 días, ese reingreso **genera** un
segundo registro en el conjunto. Reconocer al paciente equivale entonces a conocer el desenlace.

In [ ]:
X, y, grupos = cons.matriz(trabajo)
X_ent, X_eva, y_ent, y_eva, g_ent, g_eva = part.particionar(X, y, grupos)

print(f"predictoras   : {X.shape[1]}")
print(f"entrenamiento : {len(X_ent):,} filas   tasa {y_ent.mean():.4f}")
print(f"evaluacion    : {len(X_eva):,} filas   tasa {y_eva.mean():.4f}")
print("fuga de pacientes:", part.verificar_particion(g_ent, g_eva) or "ninguna")

### Creación del modelo de clasificación

Se entrena un **bosque aleatorio**: un conjunto de árboles de decisión ajustados sobre muestras
distintas de los datos, cuyo voto se promedia. Reduce la varianza de un árbol individual y maneja
sin dificultad la mezcla de variables numéricas y categóricas de este conjunto.

El umbral de decisión **no se hereda de `predict()`**. Con 11,4 % de positivos, casi ningún
paciente supera 0,5 y el clasificador termina marcando a muy pocos. Se elige por validación sobre
una porción reservada del entrenamiento, maximizando F2, y se fija en el artefacto.

In [ ]:
TECNICA = "class_weight"   # reemplazar por la ganadora de src/models/balanceo.py

bosque = ent.armar_estimador(
    X, ent.CATALOGO["bosque"]["constructor"](class_weight="balanced_subsample"), TECNICA
)

umbral = ent.elegir_umbral(bosque, X_ent, y_ent, g_ent, esq.CRITERIO_UMBRAL)
bosque.fit(X_ent, y_ent)

probabilidades = bosque.predict_proba(X_eva)[:, 1]
y_predicho = (probabilidades >= umbral).astype(int)

print(f"umbral elegido por {esq.CRITERIO_UMBRAL}: {umbral:.3f}")

### La matriz de confusión

Es el instrumento central de este análisis. Cada cuadrante tiene una lectura operativa distinta y
sus costos no son intercambiables.

In [ ]:
resultados = met.evaluar(y_eva, probabilidades, umbral)
conteos = met.matriz_confusion(y_eva, y_predicho)

vp, fp = conteos["verdaderos_positivos"], conteos["falsos_positivos"]
fn, vn = conteos["falsos_negativos"], conteos["verdaderos_negativos"]

m = np.array([[vn, fp], [fn, vp]])
etiquetas = np.array([
    [f"Verdadero negativo\n{vn:,}\nalta sin control, correcto",
     f"Falso positivo\n{fp:,}\nseguimiento gastado"],
    [f"FALSO NEGATIVO\n{fn:,}\nreingresa sin control",
     f"Verdadero positivo\n{vp:,}\nreingreso anticipado"],
]).astype(str)

fig, ax = plt.subplots(figsize=(7.2, 5.4))
sns.heatmap(m, annot=etiquetas, fmt="", cmap="Blues", cbar=False,
            linewidths=2, linecolor=SUPERFICIE,
            annot_kws={"fontsize": 10}, ax=ax)

# El cuadrante del error caro se resalta en rojo.
ax.add_patch(plt.Rectangle((0, 1), 1, 1, fill=False, edgecolor=ALERTA, linewidth=3.5))

ax.set_xticklabels(["Predice: no reingresa", "Predice: reingresa"], fontsize=9.5)
ax.set_yticklabels(["Realidad: no reingresa", "Realidad: reingresa"], fontsize=9.5, rotation=0)
ax.set_title(f"Matriz de confusión — umbral {umbral:.3f}\n"
             f"En rojo, el error caro: {fn:,} pacientes que reingresan sin seguimiento",
             loc="left", pad=14, fontsize=10.5)
fig.tight_layout()
plt.show()

#### Lectura operativa de cada cuadrante

In [ ]:
total_positivos = int(y_eva.sum())

lectura = pd.DataFrame([
    {"cuadrante": "Verdadero positivo", "casos": vp,
     "significado": "reingreso anticipado; el seguimiento llega a quien lo necesita"},
    {"cuadrante": "Falso positivo", "casos": fp,
     "significado": "se gasta un cupo de seguimiento en quien no iba a reingresar"},
    {"cuadrante": "FALSO NEGATIVO", "casos": fn,
     "significado": "el paciente sale sin control y reingresa — error caro"},
    {"cuadrante": "Verdadero negativo", "casos": vn,
     "significado": "alta sin seguimiento, correctamente"},
])
display(lectura)

print(f"\nDe los {total_positivos:,} pacientes que reingresaron:")
print(f"   {vp:,} fueron detectados     ({vp/total_positivos:.1%})")
print(f"   {fn:,} se escaparon          ({fn/total_positivos:.1%})  <- falsos negativos")
print(f"\nSeguimientos programados: {vp+fp:,}")
print(f"   de los cuales acertados: {vp:,}  ({resultados['precision']:.1%})")
print(f"   seguimientos por acierto: {resultados['seguimientos_por_acierto']}")

#### Métricas derivadas

In [ ]:
tabla = pd.DataFrame([
    ("Sensibilidad (recall)", resultados["sensibilidad"], "fracción de reingresos detectados"),
    ("Precisión", resultados["precision"], "fracción de aciertos entre los marcados"),
    ("Especificidad", resultados["especificidad"], "fracción de no-reingresos bien clasificados"),
    ("F2", resultados["f2"], "pesa el doble la sensibilidad — métrica principal"),
    ("F1", resultados["f1"], "equilibrio entre precisión y sensibilidad"),
    ("Exactitud balanceada", resultados["exactitud_balanceada"], "promedio de sensibilidad y especificidad"),
    ("Exactitud", resultados["exactitud"], "engañosa aquí: el trivial alcanza 0,886"),
    ("ROC-AUC", resultados["roc_auc"], "separación, independiente del umbral"),
    ("Brier", resultados["brier"], "calidad de la probabilidad que usa el tablero"),
], columns=["métrica", "valor", "lectura"])
display(tabla.style.format({"valor": "{:.4f}"}).hide(axis="index"))

### El umbral: la palanca sobre los falsos negativos

Bajar el umbral marca más pacientes, reduce los falsos negativos y aumenta los falsos positivos.
El gráfico muestra ese intercambio y dónde queda el punto elegido.

In [ ]:
barrido = met.barrido_umbral(y_eva, probabilidades)

fig, ejes = plt.subplots(1, 2, figsize=(10.4, 3.6))

for col, color, etq in [("sensibilidad", SERIE, "Sensibilidad"),
                        ("precision", "#8a8a8a", "Precisión"),
                        ("f2", ALERTA, "F2")]:
    ejes[0].plot(barrido["umbral"], barrido[col], color=color, linewidth=1.7, label=etq)
ejes[0].axvline(umbral, color="#9a9992", linestyle=(0, (4, 3)), linewidth=1)
ejes[0].set_xlabel("Umbral de decisión")
ejes[0].set_title("Precisión, sensibilidad y F2", loc="left")
ejes[0].legend(frameon=False, fontsize=8)

ejes[1].plot(barrido["umbral"], barrido["falsos_negativos"], color=ALERTA,
             linewidth=1.7, label="Falsos negativos")
ejes[1].plot(barrido["umbral"], barrido["falsos_positivos"], color="#8a8a8a",
             linewidth=1.7, label="Falsos positivos")
ejes[1].axvline(umbral, color="#9a9992", linestyle=(0, (4, 3)), linewidth=1)
ejes[1].set_xlabel("Umbral de decisión")
ejes[1].set_ylabel("Pacientes")
ejes[1].set_title("Los dos errores, en número de pacientes", loc="left")
ejes[1].legend(frameon=False, fontsize=8)

for ax in ejes:
    ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
plt.show()

### Preguntas de análisis

### 1. ¿Por qué no se evalúa con exactitud?

In [ ]:
from sklearn.dummy import DummyClassifier

trivial = DummyClassifier(strategy="prior").fit(X_ent, y_ent)
p_trivial = trivial.predict_proba(X_eva)[:, 1]
r_trivial = met.evaluar(y_eva, p_trivial, 0.5)

comparacion = pd.DataFrame([
    {"modelo": "Trivial (siempre 'no reingresa')", **{k: r_trivial[k] for k in
     ("exactitud", "sensibilidad", "f2", "falsos_negativos")}},
    {"modelo": "Bosque aleatorio", **{k: resultados[k] for k in
     ("exactitud", "sensibilidad", "f2", "falsos_negativos")}},
])
display(comparacion.style.format({"exactitud": "{:.4f}", "sensibilidad": "{:.4f}",
                                  "f2": "{:.4f}"}).hide(axis="index"))

**Respuesta.** El clasificador trivial alcanza una exactitud de 0,886 sin detectar a un solo
paciente en riesgo: deja escapar a **todos** los que reingresan. El bosque tiene una exactitud
*menor* y sin embargo es el único útil. La exactitud premia acertar sobre la clase mayoritaria,
que es precisamente la que no interesa.

### 2. ¿Cómo cambia el desempeño según la técnica de balanceo?

In [ ]:
# Resultados producidos por: python -m src.models.balanceo# Se leen de MLflow para que el cuaderno y las corridas no se desincronicen.import mlflowmlflow.set_tracking_uri(URI_MLFLOW)cliente = mlflow.MlflowClient()exp = cliente.get_experiment_by_name("reingreso-30d-balanceo")corridas = mlflow.search_runs([exp.experiment_id])vista = corridas[corridas["params.desbalance"].notna()][[    "params.desbalance", "metrics.umbral", "metrics.f2", "metrics.sensibilidad",    "metrics.precision", "metrics.falsos_negativos", "metrics.falsos_positivos"]]vista.columns = ["técnica", "umbral", "F2", "sensibilidad", "precisión", "FN", "FP"]display(vista.sort_values("F2", ascending=False).style.hide(axis="index"))

### 3. ¿Cuántos falsos negativos se evitan y a qué costo?

In [ ]:
costos = met.umbral_de_minimo_costo(
    y_eva, probabilidades,
    costo_falso_positivo=esq.COSTO_FALSO_POSITIVO,
    costo_falso_negativo=esq.COSTO_FALSO_NEGATIVO,
)
optimo = costos.loc[costos["costo_total"].idxmin()]

fig, ax = plt.subplots(figsize=(6.4, 3.4))
ax.plot(costos["umbral"], costos["costo_total"], color=TINTA, linewidth=1.7, label="Costo total")
ax.plot(costos["umbral"], costos["costo_falsos_negativos"], color=ALERTA,
        linewidth=1.3, linestyle="--", label="Falsos negativos")
ax.plot(costos["umbral"], costos["costo_falsos_positivos"], color="#8a8a8a",
        linewidth=1.3, linestyle="--", label="Falsos positivos")
ax.axvline(optimo["umbral"], color=SERIE, linewidth=1.2)
ax.text(optimo["umbral"], ax.get_ylim()[1]*0.95, f"  mínimo en {optimo['umbral']:.2f}",
        fontsize=8, color=SERIE)
ax.set_xlabel("Umbral de decisión")
ax.set_ylabel(f"Costo (FN vale {esq.COSTO_FALSO_NEGATIVO:.0f}x un FP)")
ax.set_title("Costo total de los errores según el umbral", loc="left")
ax.legend(frameon=False, fontsize=8)
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
plt.show()

print(f"umbral de mínimo costo : {optimo['umbral']:.3f}")
print(f"umbral elegido por F2  : {umbral:.3f}")

**Interpretación.** La razón cinco a uno entre los dos errores es un parámetro explícito, no una
constante escondida. Cada institución tiene su propia relación entre el costo de un control
posterior al alta y el de una readmisión, y el umbral óptimo se mueve con ella. Que el umbral
elegido por F2 quede cerca del de mínimo costo indica que la elección de métrica es coherente con
la estructura de costos declarada.

### 4. Observaciones y conclusiones

*(a completar con los resultados definitivos de la búsqueda de hiperparámetros)*